In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler



### Dataset setup

In [2]:
data = {
    "Age": [22, 28, np.nan, 41, 35, 26],
    "Salary": [32000, 48000, 55000, np.nan, 42000, 60000],
    "Department": ["IT", "Finance", "IT", "Marketing", np.nan, "Finance"],
    "Years_of_Experience": [1, 4, 2, 12, 7, np.nan],
}

df = pd.DataFrame(data)

numeric_features = ["Age", "Salary", "Years_of_Experience"]
categorical_features = ["Department"]

### Categorical preprocessing

In [3]:
cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

### 1. StandardScaler pipeline

In [4]:
pipe_std = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        ("cat", cat_pipe, categorical_features),
    ]
)

### 2. MinMaxScaler pipeline

In [5]:
pipe_minmax = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", MinMaxScaler()),
                ]
            ),
            numeric_features,
        ),
        ("cat", cat_pipe, categorical_features),
    ]
)



### Fit and transform the dataset

In [6]:
std_array = pipe_std.fit_transform(df)
minmax_array = pipe_minmax.fit_transform(df)

### Get feature names

In [7]:
cols = pipe_std.get_feature_names_out()

### Convert transformed arrays into DataFrames

In [8]:
df_std = pd.DataFrame(std_array, columns=cols)
df_minmax = pd.DataFrame(minmax_array, columns=cols)

### Display StandardScaler output

In [9]:
print("=== Array Output: StandardScaler ===")
print(df_std.round(4))

=== Array Output: StandardScaler ===
   num__Age  num__Salary  num__Years_of_Experience  cat__Department_Finance  \
0   -1.2810      -1.7267                   -1.0954                      0.0   
1   -0.3203       0.0557                   -0.2739                      1.0   
2   -0.3203       0.8355                   -0.8216                      0.0   
3    1.7614       0.0557                    1.9170                      0.0   
4    0.8006      -0.6127                    0.5477                      1.0   
5   -0.6405       1.3925                   -0.2739                      1.0   

   cat__Department_IT  cat__Department_Marketing  
0                 1.0                        0.0  
1                 0.0                        0.0  
2                 1.0                        0.0  
3                 0.0                        1.0  
4                 0.0                        0.0  
5                 0.0                        0.0  


### Display MinMaxScaler output

In [10]:
print("\n=== Array Output: MinMaxScaler ===")
print(df_minmax.round(4))



=== Array Output: MinMaxScaler ===
   num__Age  num__Salary  num__Years_of_Experience  cat__Department_Finance  \
0    0.0000       0.0000                    0.0000                      0.0   
1    0.3158       0.5714                    0.2727                      1.0   
2    0.3158       0.8214                    0.0909                      0.0   
3    1.0000       0.5714                    1.0000                      0.0   
4    0.6842       0.3571                    0.5455                      1.0   
5    0.2105       1.0000                    0.2727                      1.0   

   cat__Department_IT  cat__Department_Marketing  
0                 1.0                        0.0  
1                 0.0                        0.0  
2                 1.0                        0.0  
3                 0.0                        1.0  
4                 0.0                        0.0  
5                 0.0                        0.0  


### Numerical range comparison

In [11]:
num_cols = [f"num__{c}" for c in numeric_features]

print("\n=== Numerical Range Comparison ===")

for col, name in zip(num_cols, numeric_features):
    print(
        f"Feature: {name:20} | "
        f"StandardScaler Range: [{df_std[col].min():.2f}, {df_std[col].max():.2f}] | "
        f"MinMaxScaler Range: [{df_minmax[col].min():.2f}, {df_minmax[col].max():.2f}]"
    )


=== Numerical Range Comparison ===
Feature: Age                  | StandardScaler Range: [-1.28, 1.76] | MinMaxScaler Range: [0.00, 1.00]
Feature: Salary               | StandardScaler Range: [-1.73, 1.39] | MinMaxScaler Range: [0.00, 1.00]
Feature: Years_of_Experience  | StandardScaler Range: [-1.10, 1.92] | MinMaxScaler Range: [0.00, 1.00]
